# project : Hormuz crisis
- author : Christine Plumejeaud-Perreau, UMR 7301 Migrinter
- context : Master M2 Geospatial and web development with Python
- date : 18.11.2026

Processing data in [github JE-Ormuz](https://github.com/magisAR9/JE-Ormuz/blob/main/DATA.md) 


In [4]:
# Ports https://github.com/magisAR9/JE-Ormuz/raw/refs/heads/main/PORTS_monde.xlsx

import pandas as pd 
import geopandas as gpd 

rawPath = 'https://github.com/magisAR9/JE-Ormuz/raw/refs/heads/main/PORTS_monde.xlsx'
 
ports = pd.read_excel(rawPath, engine='openpyxl')

In [5]:
geoports = gpd.read_file(rawPath,  sheet_name='PORTS_monde', geometry=gpd.points_from_xy(ports['LongitudeDecimal'], ports['LatitudeDecimal']), crs="EPSG:4326") 


c:\Users\kolla\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver XLSX does not support open option SHEET_NAME
  return ogr_read(
c:\Users\kolla\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver XLSX does not support open option GEOMETRY
  return ogr_read(
c:\Users\kolla\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver XLSX does not support open option CRS
  return ogr_read(


In [6]:
geoports 

,PortID,PortName,Country,LatitudeDecimal,LongitudeDecimal
0,10003,Funafuti,Tuvalu,-8.516666,179.216667
1,10004,Famagusta,Cyprus,35.131917,33.944950
2,10005,Withnell Bay,Australia,-20.584350,116.760383
3,10006,Crinan Canal,United Kingdom,56.091433,-5.556000
4,10008,Porthmadog,United Kingdom,52.923400,-4.128850
...,...,...,...,...,...
5722,464933,Kalajoki,Finland,64.227667,23.715750
5723,464934,Cee,Spain,42.945000,-9.178333
5724,465064,Enayam,India,8.166667,77.250000
5725,465273,Puerto Morelos,Mexico,20.840783,86.880250


In [7]:
# 2. Convertir le DataFrame en GeoDataFrame
# Attention : dans gpd.points_from_xy(), on met d'abord la LONGITUDE (X), puis la LATITUDE (Y)
geoports = gpd.GeoDataFrame(
    ports,
    geometry=gpd.points_from_xy(ports["LongitudeDecimal"], ports["LatitudeDecimal"]),
    crs="EPSG:4326",  # Système de coordonnées GPS standard (WGS 84)
)

# 3. Vérifier le résultat
print(geoports.head())

   PortID      PortName         Country  LatitudeDecimal  LongitudeDecimal  \
0   10003      Funafuti          Tuvalu        -8.516666        179.216667   
1   10004     Famagusta          Cyprus        35.131917         33.944950   
2   10005  Withnell Bay       Australia       -20.584350        116.760383   
3   10006  Crinan Canal  United Kingdom        56.091433         -5.556000   
4   10008    Porthmadog  United Kingdom        52.923400         -4.128850   

                      geometry  
0   POINT (179.21667 -8.51667)  
1    POINT (33.94495 35.13192)  
2  POINT (116.76038 -20.58435)  
3      POINT (-5.556 56.09143)  
4     POINT (-4.12885 52.9234)  


## Une carte avec cartopy

In [8]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# 2. Initialiser la figure et la projection de la carte
# PlateCarree est la projection standard pour les données en lat/lon
fig, ax = plt.subplots(
    figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()}
)

# 3. Ajouter les éléments géographiques de base
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN, facecolor="aliceblue")
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.8)

# 4. Afficher les points du GeoDataFrame
# L'argument transform=ccrs.PlateCarree() indique que vos données sont en lat/lon (WGS 84)
geoports.plot(ax=ax, color="red", markersize=30, transform=ccrs.PlateCarree())

# 5. (Optionnel) Zoomer automatiquement autour de vos points avec une petite marge
minx, miny, maxx, maxy = geoports.total_bounds
marge = 1.0  # en degrés
ax.set_extent(
    [minx - marge, maxx + marge, miny - marge, maxy + marge],
    crs=ccrs.PlateCarree(),
)

# 6. Ajouter une grille de coordonnées et afficher
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
plt.title("Ports of the world (cartopy)")

# Sauvegarder la carte en image basse résolution (90 DPI)
plt.savefig(
    "ports_world.png",
    dpi=90,
    bbox_inches="tight",  # Supprime les marges blanches inutiles autour de la carte
    facecolor="white",  # Assure un fond blanc opaque derrière la figure
)
#plt.show()

# Fermer la figure pour libérer la mémoire (recommandé si vous générez plusieurs cartes)
#plt.close(fig)

ModuleNotFoundError: No module named 'cartopy'

In [ ]:
plt.close(fig)

## Another way to do it with folium

In [ ]:
# Définir les chemins d'accès aux fichiers Excel
path = "C:/Travail/Cartographie/JE_Ormuz_13nov2026/atelier/JE-Ormuz/data/"
escales_detroit = "ESCALES PC GNL ET TANKER.xlsx"
escales_monde_petroliers = "ESCALES_GDS_TANKERS_+55000dwt.xlsx" 
ports_monde = 'PORTS_monde.xlsx'


In [ ]:
escales_petroliers_W = pd.read_excel(path+escales_monde_petroliers)
ports_W = pd.read_excel(path+ports_monde) 
ports_detroit_ormuz = pd.read_excel(path+escales_detroit, sheet_name='PORTS')


In [ ]:
import xyzservices
import folium
import numpy as np



m = folium.Map(location=[ np.mean(ports_detroit_ormuz.LatitudeDecimal), np.mean(ports_detroit_ormuz.LongitudeDecimal)] )

#Ajouter les ports
longitudes = np.array(ports_detroit_ormuz.LongitudeDecimal)
latitudes = np.array(ports_detroit_ormuz.LatitudeDecimal)
names = np.array(ports_detroit_ormuz.Port)


for i in range(len(latitudes)):
    if names[i] == 'Al Jazeera Port ' : 
        folium.CircleMarker(
            location=[latitudes[i], longitudes[i]],
            radius=10,
            popup=names[i],
            color="#7931cc",
            fill=True,
            fill_color="#7931cc").add_to(m)
    else : 
        folium.CircleMarker(
            location=[latitudes[i], longitudes[i]],
            radius=5,
            popup=names[i],
            color="#cc3198",
            fill=True,
            fill_color="#ee0996",
        ).add_to(m)

m

m.save("map_detroit_port_ORMUZ.html")

### Only ports around Hormuz


In [ ]:
import numpy as np
ports_detroit_ormuz.head()
ports_detroit_ormuz.shape #(64, 4)

print(ports_detroit_ormuz.columns) #['Port', 'Country', 'LatitudeDecimal', 'LongitudeDecimal']

#centrer sur LatitudeDecimal LongitudeDecimal
np.mean(ports_detroit_ormuz.LatitudeDecimal)
np.mean(ports_detroit_ormuz.LongitudeDecimal)

## Map of call port for tankers

1. use the api to get the data
2. open the file and the good sheet
3. make the join with the geoports 
4. build a map whose circles size is proportionnal to the number of calls
5. Then, distinguish between dates of call, and make a map with green circles before Ormuz, and red circles after the crisis
6. Build an histogram for one choosen OD (Origin-Destination), with time duration, before, and after the crisis.
7. Track the ships that were around Hormuz at the beginning of the crisis : what trajectory have they, how long do they have beeing waiting for, how much of them change their route ?
8. Build the geometry of the route anf plot it, for one typical ship.